# Custom dataset: study design, load a CSV, train an AI model, simulate with CoXAM

The other tutorials use one of the 9 datasets bundled under
`assets/original_datasets/` (`adult`, `wine_quality`, `mushrooms`, ...),
looked up by a fixed `dataset_id`. This notebook uses `prepare_dataset()`'s
other mode instead: point it at a CSV directly and it builds the same kind
of prepared dataset, without adding anything to that fixed registry.

**This is deliberately ad hoc.** `dataset_id` here (`"creditcard_sample"`)
is just a display label you choose -- it is never looked up or saved
anywhere. Nothing is written to disk and no new module is registered, so a
fresh session has to pass `csv_path`/`dataframe`/`target_col` again to
rebuild the same dataset; there is no `prepare_dataset(dataset_id=
"creditcard_sample")` shortcut later. If you want that permanence, the
separate path is adding a real `assets/original_datasets/<name>/load.py`
and registering it -- not what this notebook does.

The dataset: a sample of the published Kaggle "Credit Card Fraud
Detection" data -- all 492 fraudulent transactions plus 3,000 randomly
sampled legitimate ones (see Phase 1.3 for why the full 284,807 rows are
downsampled first). `V1`-`V28` are PCA components of the original
transaction features, which the dataset's own publishers never
disclosed; only `Time` and `Amount` keep real names.

The cognitive model: **CoXAM**, run with `source="fit"` -- it fits fresh
decision-tree (`"Rules"`) and logistic-regression (`"Weights"`) surrogates
directly against this study's own trained model, rather than reading
CoXAM's published corpus (which only covers `wine_quality`/`mushrooms`).
Per its own docstring, `source="fit"` "works for any dataset and any AI
model" -- that claim is what this notebook checks. It also fits its own
surrogates internally, so there is no separate `xaikitTest.explanations()`
call before simulating, unlike the general-workflow tutorial.

## Phase 0: Set up your environment

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Walk up until the repo root (the directory holding `src/`) is found, so the
# notebook runs from anywhere inside the checkout.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src as xk

OUTPUT_DIR = REPO_ROOT / "tutorials" / "custom_dataset_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = REPO_ROOT / "assets" / "ai_dataset" / "other" / "creditcard.csv"
print("repo root :", REPO_ROOT)
print("output dir:", OUTPUT_DIR)
print("csv path  :", CSV_PATH, "exists:", CSV_PATH.is_file())

repo root : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api
output dir: /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/custom_dataset_output
csv path  : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/assets/ai_dataset/other/creditcard.csv exists: True


## Phase 1: Configure the User Study

### Step 1.1 Initialize the study workflow

In [2]:
xaikitTest = xk.xaikitTest("creditcard_custom_coxam_study", output_dir=OUTPUT_DIR)
xaikitTest

### Step 1.2 Define the user study variables

Same shape CoXAM's own tutorial uses: `xai_type` (which explanation
family, `decision_tree` vs `logistic_regression`), within-subjects and
block-randomized, and `tested_w_xai` (whether the trial shows an
explanation at all), within-subjects and randomized per trial. The DV,
`forward_accuracy`, is whether the simulated participant predicted the
AI's own output correctly.

In [3]:
iv_config = {
    "xai_type": {
        "type": "within",
        "randomization": "block",
        "levels": ["decision_tree", "logistic_regression"],
    },
    "tested_w_xai": {
        "type": "within",
        "randomization": "trial",
        "levels": [True, False],
    },
}

dvs = {"forward_accuracy": ["continuous"]}

xaikitTest.set_design(iv_config=iv_config, dvs=dvs, show=True)
xaikitTest.validate(stage="design", strict=False, show=True)


IV configuration:
  xai_type             type=within   randomization=block levels=['decision_tree', 'logistic_regression']
  tested_w_xai         type=within   randomization=trial levels=[True, False]

CVs: []
DVs: ['forward_accuracy']
XAIKit validation: design
Good evening.

Reminders:
  - user_task: Set `user_task` for cognitive simulation.
    Suggestion: Planner-only OK. Supported: counterfactual_simulation, forward_simulation.


XAIKit validation: design
Good evening.

Reminders:
  - user_task: Set `user_task` for cognitive simulation.
    Suggestion: Planner-only OK. Supported: counterfactual_simulation, forward_simulation.

Normalized:
  IVs:
    xai_type: decision_tree, logistic_regression
    tested_w_xai: True, False
  DVs: forward_accuracy
  Context:
    dataset_id: not set
    model_name: not set
    model_source: not set
    cognitive_model_id: placeholder
    tasks: not set
    xai_methods: not set
    xai_types: decision_tree, logistic_regression
    xai_faithfulness: not set
    dvs: forward_accuracy

### Step 1.3 Load the custom dataset

The full CSV is 284,807 rows, only 492 of them (0.17%) fraud. CoXAM's
`source="fit"` fits its decision-tree/logistic-regression surrogates
against **every** row in the prepared split, not a capped sample --
reasonable for CoXAM's own published corpora (a few thousand rows) but
impractically slow at the full size here (measured: ~14 minutes for 96
trials). Keeping every fraud case and randomly sampling 3,000 legitimate
ones brings that under 10 seconds with the class of interest fully
represented -- at the cost of no longer reflecting the real 0.17% base
rate (about 14% fraud in the sample instead).

`target_col="Class"` is already binary (0/1); `positive_class=1` fixes
which value is the positive class rather than leaving it to whichever
value the loader sees first. `cognitive_model_id="coxam"` selects the top
6 features by correlation with the target -- CoXAM's published corpus is
always 6-feature, so a freshly trained model matches that shape.

In [4]:
full = pd.read_csv(CSV_PATH)
fraud = full[full["Class"] == 1]
legitimate_sample = full[full["Class"] == 0].sample(n=3000, random_state=42)
sample = (
    pd.concat([fraud, legitimate_sample])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print(f"sample: {len(sample)} rows, {(sample['Class'] == 1).sum()} fraud")

data = xaikitTest.prepare_dataset(
    dataset_id="creditcard_sample",   # a label only -- not looked up or saved anywhere
    dataframe=sample,
    target_col="Class",
    positive_class=1,
    cognitive_model_id="coxam",       # -> top 6 features by target correlation
    show_available=True,
    show_summary=True,
)

print()
print("selected features:", data.feature_names)

sample: 3492 rows, 492 fraud
Available training datasets: ['creditcard_sample' (custom)]
Dataset   : creditcard_sample  (3492 rows, 6 model features)
Features  : ['V14', 'V12', 'V11', 'V10', 'V4', 'V16']
Encoding  : one-hot
Train set : 2793 samples  (80%)
Test set  : 699 samples  (20%)
Class balance (train) -> class 0: 2399
Class balance (train) -> class 1: 394
First test instanceIds: [2940, 1312, 1316, 1228, 2400, 1276, 3375, 631, 1549, 1175]

selected features: ['V14', 'V12', 'V11', 'V10', 'V4', 'V16']


## Phase 2: Prepare the AI Model

Same call as any bundled dataset -- `prepare_dataset()` already returned a
normal `PreparedDataset`, so nothing downstream needs to know this one
came from a dataframe instead of `assets/original_datasets/`. No
`xaikitTest.explanations()` call here: CoXAM's `source="fit"` (Phase 3)
fits its own surrogates directly against this trained model.

In [5]:
xaikitTest.train_AI_model(
    model_type="mlp",
    batch_size=64,
    train_kwargs={"epochs": 5},
)

metrics = xaikitTest.evaluate(split="test")
pd.DataFrame(metrics).T[["accuracy", "roc_auc", "precision_macro", "recall_macro"]]

,accuracy,roc_auc,precision_macro,recall_macro
test,0.974249,0.966756,0.980126,0.912433


## Phase 3: Run the User Study Simulation

### Step 3.1 Build the trial table

`xai_type` and `tested_w_xai` are both within-subjects, so every
participant sees both explanation families -- 2 blocks x 2 trial-level
levels = 4 balanced cells, so `num_testing` must divide evenly by 4.

In [6]:
n_block_conditions = len(iv_config["xai_type"]["levels"])
n_trial_conditions = len(iv_config["tested_w_xai"]["levels"])
n_balanced_cells = n_block_conditions * n_trial_conditions

participants_per_between_condition = 4
num_training = 4
num_testing = 8

assert num_testing % n_balanced_cells == 0, (
    "num_testing must divide evenly across block x trial-level cells: "
    f"{num_testing} trials for {n_balanced_cells} cells"
)

trial_result = xaikitTest.generate_trials(
    participants_per_between_condition=participants_per_between_condition,
    num_training=num_training,
    num_testing=num_testing,
    output_dir="trials",
    show=True,
)

pd.DataFrame(trial_result.trials).head(12)

Counterbalancing strategy: complete_counterbalancing
Participant assignments: 4 total
Instance pool rows: 300
Trial rows: 48
Exported trial artifacts:
  CSV     : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/custom_dataset_output/trials/trials.csv
  JSON    : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/custom_dataset_output/trials/trials.json
  Summary : /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/custom_dataset_output/trials/design_summary.json

Previewing first 10 trial rows:
{'participantId': 1, 'trialId': 1, 'phase': 'training', 'phaseTrialId': 1, 'block': 1, 'trialWithinBlock': 1, 'withinCondition': 'decision_tree', 'xai_type': 'decision_tree', 'dataId': 'creditcard_sample', 'instanceId': '540'}
{'participantId': 1, 'trialId': 2, 'phase': 'training', 'phaseTrialId': 2, 'block': 1, 'trialWithinBlock': 2, 'withinCondition': 'decision_tree', 'xai_type': 'decision_tree', 'dataId': 'creditcard_sample', 'instanceId': '364'

,participantId,trialId,block,trialWithinBlock,withinCondition,xai_type,dataId,instanceId,phase,phaseTrialId,shown_xai_type,tested_w_xai
0,1,1,1,1,decision_tree,decision_tree,creditcard_sample,540,training,1,decision_tree,NaN
1,1,2,1,2,decision_tree,decision_tree,creditcard_sample,364,training,2,decision_tree,NaN
2,1,3,2,1,logistic_regression,logistic_regression,creditcard_sample,2262,training,3,logistic_regression,NaN
3,1,4,2,2,logistic_regression,logistic_regression,creditcard_sample,3184,training,4,logistic_regression,NaN
4,1,5,1,1,decision_tree,decision_tree,creditcard_sample,1424,testing,1,decision_tree,False
5,1,6,1,2,decision_tree,decision_tree,creditcard_sample,693,testing,2,decision_tree,True
6,1,7,1,3,decision_tree,decision_tree,creditcard_sample,767,testing,3,decision_tree,False
7,1,8,1,4,decision_tree,decision_tree,creditcard_sample,469,testing,4,decision_tree,True
8,1,9,2,1,logistic_regression,logistic_regression,creditcard_sample,2909,testing,5,logistic_regression,False
9,1,10,2,2,logistic_regression,logistic_regression,creditcard_sample,3022,testing,6,logistic_regression,False


### Step 3.2 Register CoXAM and run the simulation

`source="fit"` is the default -- passing it explicitly here just makes
the choice visible. It fits fresh decision-tree/logistic-regression
surrogates against `xaikitTest.data`/`xaikitTest.trained_ai_model`
directly, so it works on this custom dataset exactly as it would on any
bundled one.

In [7]:
xaikitTest.set_cognitive_model(cognitive_model_id="coxam")
print(f"Cognitive model: {xaikitTest.cognitive_model_id}")

simulated_results = xaikitTest.run_experiment(
    mode="whole_experiment",
    source="fit",
)
testing_results = simulated_results.query("phase == 'testing'").copy()

print(f"{len(xaikitTest.trials):,} trials -> {len(simulated_results):,} recorded rows")
display(
    simulated_results
    .groupby(["phase", "xai_type", "tested_w_xai"], dropna=False)
    .size()
    .reset_index(name="rows")
)
simulated_results.head()

Cognitive model: coxam


48 trials -> 48 recorded rows


,phase,xai_type,tested_w_xai,rows
0,testing,decision_tree,False,8
1,testing,decision_tree,True,8
2,testing,logistic_regression,False,8
3,testing,logistic_regression,True,8
4,training,decision_tree,NaN,8
5,training,logistic_regression,NaN,8


,forward_accuracy,participantId,trialId,block,trialWithinBlock,withinCondition,xai_type,dataId,instanceId,phase,...,tested_w_xai,step,condition_name,explanation_type,selected_strategy,ai_prediction,agent_prediction,cognitive_correct_vs_ai,prob_correct,pred_time
0,1,1,1,1,1,decision_tree,decision_tree,creditcard_sample,540,training,...,NaN,0,decision_tree,dt,dt_traversal,0,0,True,0.875000,17.861779
1,1,1,2,1,2,decision_tree,decision_tree,creditcard_sample,364,training,...,NaN,1,decision_tree,dt,dt_traversal,0,0,True,0.937500,17.861779
2,0,1,3,2,1,logistic_regression,logistic_regression,creditcard_sample,2262,training,...,NaN,0,linear_regression,lr,lr_heuristic,0,1,False,0.437569,1.599339
3,1,1,4,2,2,logistic_regression,logistic_regression,creditcard_sample,3184,training,...,NaN,1,linear_regression,lr,lr_heuristic,0,0,True,0.687293,1.599339
4,1,1,5,1,1,decision_tree,decision_tree,creditcard_sample,1424,testing,...,False,2,decision_tree,none,dt_traversal,0,0,True,0.625000,5.338659


## Phase 4: Analysis of Simulation Results

### Step 4.1 Descriptive statistics

In [8]:
analysis = xaikitTest.analyze_iv_dv(
    iv="xai_type",
    dv="forward_accuracy",
)
print("--- xai_type x forward_accuracy ---")
display(analysis.descriptives)

--- xai_type x forward_accuracy ---


/Users/wangzhuoyulucas/anaconda3/envs/xaik-api-dev/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,index,xai_type,count,mean,std,sem
0,0,decision_tree,4,0.750,0.000000,0.000000
1,1,logistic_regression,4,0.875,0.144338,0.072169


## What this notebook showed

*   `prepare_dataset(csv_path=.../dataframe=..., target_col=...)` builds
    the exact same `PreparedDataset` a bundled `dataset_id` does --
    `train_AI_model()` and `generate_trials()` run unchanged.
*   `cognitive_model_id="coax"`/`"coxam"` auto-selects 5/6 features by
    target correlation for a dataset with no curated feature list of its
    own.
*   **CoXAM genuinely runs against a custom dataset** via
    `run_experiment(source="fit")` -- no separate explanation-generation
    step, no published corpus required. The catch is scale: fitting
    against the full 284,807-row dataset took ~14 minutes for 96 trials;
    the same run on a ~3,500-row sample took ~7 seconds. `source="fit"`
    fits its surrogates against every row of the prepared split, not a
    capped sample, so dataset size -- not trial count -- is what to
    watch when trying this on a new custom dataset.
*   This dataset is **not** persisted: `dataset_id="creditcard_sample"`
    here is just a label, not a registration. A future session repeats
    `dataframe=...`/`csv_path=...` to rebuild it.